# Radiology Report Agent - Strands Deployment

This notebook demonstrates deploying a Strands-based radiology report validation agent using Amazon Bedrock Agents.

## Architecture Benefits
- **50-70% cost reduction** vs traditional Bedrock Agents
- **2-3x performance improvement** with direct tool execution
- **No Lambda functions** required

## Tools Available
- `check_radiology_report`: Validates medical accuracy and completeness
- `identify_anatomical_structures`: Identifies and categorizes anatomical references

## 1. Setup and Configuration

Configure the Bedrock AgentCore runtime with Strands entrypoint.

In [2]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
print(f"Region: {region}")

agentcore_runtime = Runtime()
agent_name = "radiology_report_validator"

response = agentcore_runtime.configure(
    entrypoint="deploy_strands.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    region=region,
    agent_name=agent_name
)
response

Entrypoint parsed: file=/home/ubuntu/tools/amazon-bedrock-agents-healthcare-lifesciences/agents_catalog/09-Radiology-Report-Agent/deploy_strands.py, bedrock_agentcore_name=deploy_strands
Configuring BedrockAgentCore agent: radiology_report_validator


Region: us-east-2


⚠️  [WARNING] Platform mismatch: Current system is 'linux/amd64' but Bedrock AgentCore requires 'linux/arm64'.
For deployment options and workarounds, see: 
https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html

Generated Dockerfile: /home/ubuntu/tools/amazon-bedrock-agents-healthcare-lifesciences/agents_catalog/09-Radiology-Report-Agent/Dockerfile
Generated .dockerignore: /home/ubuntu/tools/amazon-bedrock-agents-healthcare-lifesciences/agents_catalog/09-Radiology-Report-Agent/.dockerignore
Changing default agent from 'rad_validator' to 'radiology_report_validator'
Bedrock AgentCore configured: /home/ubuntu/tools/amazon-bedrock-agents-healthcare-lifesciences/agents_catalog/09-Radiology-Report-Agent/.bedrock_agentcore.yaml


ConfigureResult(config_path=PosixPath('/home/ubuntu/tools/amazon-bedrock-agents-healthcare-lifesciences/agents_catalog/09-Radiology-Report-Agent/.bedrock_agentcore.yaml'), dockerfile_path=PosixPath('/home/ubuntu/tools/amazon-bedrock-agents-healthcare-lifesciences/agents_catalog/09-Radiology-Report-Agent/Dockerfile'), dockerignore_path=PosixPath('/home/ubuntu/tools/amazon-bedrock-agents-healthcare-lifesciences/agents_catalog/09-Radiology-Report-Agent/.dockerignore'), runtime='Docker', region='us-east-2', account_id='575108919340', execution_role=None, ecr_repository=None, auto_create_ecr=True)

## 2. Deploy Agent

Deploy using CodeBuild mode (recommended) to handle cross-platform builds automatically.

**Note**: The platform warning (AMD64 vs ARM64) is expected and resolved by CodeBuild deployment.

In [3]:
# Deploy using CodeBuild (handles cross-platform builds)
print("Starting CodeBuild deployment...")
launch_result = agentcore_runtime.launch()
print(f"Deployment complete: {launch_result}")

🚀 CodeBuild mode: building in cloud (RECOMMENDED - DEFAULT)
   • Build ARM64 containers in the cloud with CodeBuild
   • No local Docker required
💡 Available deployment modes:
   • runtime.launch()                           → CodeBuild (current)
   • runtime.launch(local=True)                 → Local development
   • runtime.launch(local_build=True)           → Local build + cloud deploy (NEW)
Starting CodeBuild ARM64 deployment for agent 'radiology_report_validator' to account 575108919340 (us-east-2)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: radiology_report_validator


Starting CodeBuild deployment...
Repository doesn't exist, creating new ECR repository: bedrock-agentcore-radiology_report_validator


✅ ECR repository available: 575108919340.dkr.ecr.us-east-2.amazonaws.com/bedrock-agentcore-radiology_report_validator
Getting or creating execution role for agent: radiology_report_validator
Using AWS region: us-east-2, account ID: 575108919340
Role name: AmazonBedrockAgentCoreSDKRuntime-us-east-2-4e82eb21e3
Role doesn't exist, creating new execution role: AmazonBedrockAgentCoreSDKRuntime-us-east-2-4e82eb21e3
Starting execution role creation process for agent: radiology_report_validator
✓ Role creating: AmazonBedrockAgentCoreSDKRuntime-us-east-2-4e82eb21e3
Creating IAM role: AmazonBedrockAgentCoreSDKRuntime-us-east-2-4e82eb21e3
✓ Role created: arn:aws:iam::575108919340:role/AmazonBedrockAgentCoreSDKRuntime-us-east-2-4e82eb21e3
✓ Execution policy attached: BedrockAgentCoreRuntimeExecutionPolicy-radiology_report_validator
Role creation complete and ready for use with Bedrock AgentCore
✅ Execution role available: arn:aws:iam::575108919340:role/AmazonBedrockAgentCoreSDKRuntime-us-east-2-4e

Deployment complete: mode='codebuild' tag='bedrock_agentcore-radiology_report_validator:latest' env_vars=None port=None runtime=None ecr_uri='575108919340.dkr.ecr.us-east-2.amazonaws.com/bedrock-agentcore-radiology_report_validator' agent_id='radiology_report_validator-SYkpM2435F' agent_arn='arn:aws:bedrock-agentcore:us-east-2:575108919340:runtime/radiology_report_validator-SYkpM2435F' codebuild_id='bedrock-agentcore-radiology_report_validator-builder:423b2357-318f-4c46-803f-d31aa92ee65c' build_output=None


## 3. Test the Agent

Test both radiology report validation and anatomical structure identification.

In [6]:
# Test radiology report validation
# test_report = "Chest X-ray shows clear lung fields with no acute findings. Heart size normal."
test_query = "Transthoracic echocardiogram was performed of technically limited quality. \
    The left ventricle was normal in size and dimensions with normal LV function. Ejection fraction \
    was 50% to 55%. Concentric hypertrophy noted with interventricular septum measuring 1.6 cm, \
    posterior wall measuring 1.2 cm. Left atrium is enlarged, measuring 4.42 cm. \
    Right-sided chambers are normal in size and dimensions. Aortic root has normal diameter. \
    Mitral and tricuspid valve reveals annular calcification. Fibrocalcific valve leaflets noted \
    with adequate excursion. Similar findings noted on the aortic valve as well with \
    significantly adequate excursion of valve leaflets. Atrial and ventricular septum are intact.\
    Pericardium is intact without any effusion. No obvious intracardiac mass or thrombi noted. \
    Doppler study reveals mild-to-moderate mitral regurgitation. Severe aortic stenosis with peak \
    velocity of 2.76 with calculated ejection fraction 50% to 55% with severe aortic stenosis. There is also mitral stenosis."


validation_result = agentcore_runtime.invoke({
    "prompt": f"Please validate this radiology report: {test_query}"
})

print("Validation Result:")
print(validation_result['response'])

Validation Result:
['"Based on my knowledge of echocardiogram reporting standards, I can provide you with feedback on this report:\\n\\nThis echocardiogram report is relatively comprehensive but has some inconsistencies and areas for improvement:\\n\\nStrengths:\\n- Describes left/right ventricle and left atrium size and function\\n- Provides LV wall measurements and identifies hypertrophy\\n- Includes valvular assessment\\n- Comments on pericardium\\n- Includes Doppler findings\\n\\nAreas for improvement:\\n1. Internal inconsistency: \\"Severe aortic stenosis\\" is mentioned twice with insufficient hemodynamic data (only peak velocity of 2.76 is provided without units, gradient or valve area)\\n2. Mitral stenosis is mentioned without supporting measurements (no mitral valve area, gradient, or pressure half-time)\\n3. Technical quality limitations are mentioned but not specified\\n4. Diastolic function assessment is missing\\n5. Right ventricular function metrics (TAPSE, S\') are missi

In [5]:
# Test anatomical structure identification
anatomy_text = "CT scan of the abdomen reveals normal liver, spleen, and kidneys. No masses detected."

anatomy_result = agentcore_runtime.invoke({
    "prompt": f"Identify anatomical structures in: {anatomy_text}"
})

print("Anatomical Structures:")
print(anatomy_result['response'])

Anatomical Structures:
['I apologize for the technical difficulties with the function. Based on my analysis of the report, I can identify the following anatomical structures mentioned:\n\n1. Liver\n2. Spleen\n3. Kidneys\n\nHowever, the report is quite brief and omits several important anatomical structures that should be evaluated in a complete abdominal CT scan according to ACR guidelines, including:\n\n- Pancreas\n- Gallbladder and biliary system\n- Adrenal glands\n- Stomach and intestines\n- Vasculature (aorta, IVC, portal vein, etc.)\n- Lymph nodes\n- Retroperitoneum\n- Pelvic organs (if included in scan range)\n- Abdominal wall\n- Peritoneal cavity (for free fluid/masses)\n\nFor a comprehensive abdominal CT report that meets ACR guidelines, all visible structures should be systematically evaluated and documented. The current report is too limited in scope and does not provide sufficient detail for a complete diagnostic assessment.']


## 4. Monitoring and Observability

Access CloudWatch logs and X-Ray traces for the deployed agent.

In [7]:
# Get agent status and information
status_response = agentcore_runtime.status()
print(f"Agent Status: {status_response.endpoint['status']}")
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"Dashboard: https://console.aws.amazon.com/cloudwatch/home?region={region}#gen-ai-observability/agent-core")

Retrieved Bedrock AgentCore status for: radiology_report_validator


Agent Status: READY
Agent ARN: arn:aws:bedrock-agentcore:us-east-2:575108919340:runtime/radiology_report_validator-SYkpM2435F
Dashboard: https://console.aws.amazon.com/cloudwatch/home?region=us-east-2#gen-ai-observability/agent-core


## 5. Cleanup (Optional)

Remove the deployed agent and associated resources.

In [ ]:
# Uncomment to delete the agent
agentcore_runtime.delete()
print("Agent deleted successfully")